# Lab 48: Distributed backends and failure handling

[Lab 46](../46-scaling-the-signals/) left three stand-ins: a file lock (needs a shared filesystem), claim-before-send (a failed page keeps its slot), and raw-bytes corpus hashing (a reformat looks like a change). Make each production-shaped: a `RedisStore` behind the same interface, release-on-failure + a dead-letter queue, and normalized content hashing. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
import json
import pathlib
import sys
loop = pathlib.Path.cwd().parent / "41-operating-the-loop"
sys.path.insert(0, str(loop))
print("real backends + failure handling + content-aware change in:", loop.name)

## Step 1: A real distributed store (item 1)

`RedisStore` over the same interface; one atomic Lua claim works across serverless and multi-region workers.

In [ ]:
from store import RedisStore, FakeRedis, make_store
# TODO: build a RedisStore over a FakeRedis and show a second worker is suppressed for the
# same incident (one atomic claim). Confirm make_store('memory') returns an InMemoryStore.
raise NotImplementedError

## Step 2: Release-on-failure + dead-letter (item 2)

In [ ]:
from notify import deliver, format_alert, to_slack, DeadLetter
import os
import tempfile

import notify
from urllib.error import URLError
# TODO: force notify.post to raise URLError, call deliver with a store + dead_letter, and
# confirm (a) the status says dead-lettered + slot released, (b) the DLQ has one entry, and
# (c) a follow-up claim succeeds because the slot was freed.
raise NotImplementedError

## Step 3: Normalized content hashing (item 3)

In [ ]:
from canary import normalize_corpus_text, per_doc_fingerprint
import hashlib
# Item 3: the per-document map hashed raw bytes, so a reformat (CRLF, trailing spaces, an
# extra blank line) looked like a content change and triggered canary review. Hash the
# NORMALIZED content instead.
base     = b"# Helix\n\nAanya Rao leads Helix Lab.\n"
reformat = b"# Helix\r\n\n\nAanya Rao leads Helix Lab.   \n\n"   # cosmetic only
changed  = b"# Helix\n\nTomas Vega leads Helix Lab.\n"             # real edit
def h(blob):
    return hashlib.sha256(normalize_corpus_text(blob).encode()).hexdigest()[:12]
print("raw bytes differ on reformat:", hashlib.sha256(base).hexdigest()[:12], "vs", hashlib.sha256(reformat).hexdigest()[:12])
print("normalized hash, base vs reformat:", h(base), "==", h(reformat), "->", h(base)==h(reformat))
print("normalized hash, base vs real edit:", h(base), "!=", h(changed), "->", h(base)!=h(changed))
print("So a reformat triggers no review; a content edit still does.")

## Step 4: The cadence

In [ ]:
# The cadence is unchanged except the nightly notify now dead-letters failed pages
# (and caches the queue), so a flaky webhook never silently drops an alert. The per-document
# review picks up normalized hashing automatically (it is the default).
print("Pick the backend by config (memory | file: | redis://); failed pages are captured,")
print("not lost; and cosmetic corpus edits stop waking the canary review.")

## Step 5: The theme

In [ ]:
# The theme: the stand-ins from Lab 46 become the things you actually run.
#  - a file lock -> Redis/DB (works for serverless and multi-region);
#  - claim-before-send -> release-on-failure + a dead-letter queue (no silent drops);
#  - raw-bytes hashing -> normalized hashing (only content changes count).
print("A stand-in that needs a shared filesystem, loses failed pages, or fires on whitespace")
print("is a stand-in. Swap in the real backend, handle failure, and compare content, not bytes.")

## What you built

The production-shaped versions of Lab 46's three stand-ins: a `RedisStore` whose `try_claim` is a single atomic Lua round-trip (cooldown via a per-key value, rate limit via a global sliding-window sorted set) so the claim holds across serverless and multi-region workers, selectable through `make_store('redis://...')`; a `release` operation on every backend plus a `DeadLetter` queue, so `notify.deliver` frees the cooldown slot and records the payload when a send exhausts its retries instead of silently keeping the slot; and `normalize_corpus_text` so the per-document fingerprint ignores cosmetic reformats (line endings, trailing whitespace, blank-line runs) and fires only on real content changes.

**Where this simplifies:** `FakeRedis` mirrors Redis's single-threaded atomic execution for the demo and tests — production uses a real client and the same Lua; the dead-letter queue is an append-only file (a real one is a durable queue with redelivery and inspection); and normalized hashing is format-insensitive, not meaning-sensitive — a prose reflow that rewraps lines still changes the hash (semantic hashing via embeddings is a separate, noisier tool).

Next: [Lab 49](../49-graded-gold/) takes the evaluation anchor from binary to graded — an ordinal rubric, a real adjudication protocol, and the Lab 45 annotator weights re-derived against gold.